# Apache Spark Assignment — Week 5
### Step 1: Environment Setup & Spark Session
### Step 2: Loading the Mapped Dataset

In [1]:
import sys
!{sys.executable} -m pip install pyspark jupyter

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

# Initialize Spark locally using all available CPU cores
spark = SparkSession.builder \
    .appName("SparkBasicsAssignment") \
    .master("local[*]") \
    .getOrCreate()

# Suppress warnings and info logs to keep output clean
spark.sparkContext.setLogLevel("ERROR")
print("Spark Session initialized successfully!")

Spark Session initialized successfully!


In [3]:
# Pathing assumes your notebook is inside the 'notebook/' folder
input_path = os.path.join("..", "data", "dataset.csv")

df = spark.read.csv(input_path, header=True, inferSchema=True)
print("--- DATAFRAME SCHEMA ---")
df.printSchema()

print("\n--- DATA PREVIEW ---")
df.show(5)

--- DATAFRAME SCHEMA ---
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)


--- DATA PREVIEW ---
+--------+----------------+------+---------------+----------------+-----------+--------+--------+---+------------+------+-------------------+--------------------+--------+
| user_id|transaction_date|region|           city|product_category|sale_amount|store_id|   price|age|subscription|status|      raw_timestamp|               email|username|
+--------+----------------+----

### Q3: Removing Duplicate Rows
In real-world data engineering pipelines, we almost always have to deal with duplicates. A row might get logged twice because of network retries, system glitches, or messy manual uploads. 

For this assignment, we are explicitly told to target duplicate rows based on a specific set of matching columns: `user_id` and `transaction_date`. This means even if other columns (like `sale_amount` or `product_category`) vary slightly, Spark will treat rows with identical users on identical dates as duplicates, preserving the first occurrence and cleaning out the noise.

In [4]:
# dropDuplicates() with a column list passes an explicit constraint array to the execution engine.
# It ensures Spark isolates uniqueness based strictly on user identity and order dates.
df_dedup = df.dropDuplicates(["user_id", "transaction_date"])

# Running a count action forces Spark's lazy evaluation engine to process the data partitions.
# This prints an exact metric comparing our raw data pool against our clean data pool.
original_count = df.count()
deduplicated_count = df_dedup.count()
rows_removed = original_count - deduplicated_count

print(f"Original record count: {original_count}")
print(f"De-duplicated record count: {deduplicated_count}")
print(f"Total duplicate rows removed: {rows_removed}")

# Displaying a small slice to visually inspect our structural alignment
df_dedup.select("user_id", "transaction_date", "region", "product_category").show(5, truncate=False)

Original record count: 9994
De-duplicated record count: 4992
Total duplicate rows removed: 5002
+--------+----------------+-------+----------------+
|user_id |transaction_date|region |product_category|
+--------+----------------+-------+----------------+
|AA-10315|10/4/2015       |West   |Office Supplies |
|AA-10315|3/3/2016        |Central|Office Supplies |
|AA-10315|3/31/2014       |West   |Office Supplies |
|AA-10315|6/29/2017       |Central|Office Supplies |
|AA-10315|9/15/2014       |East   |Office Supplies |
+--------+----------------+-------+----------------+
only showing top 5 rows


### Q4: Filtering & Aggregating Regional Sales
In this step, we are isolating our data to look only at the `West` region, and then breaking down the average order size for each product category. 

Writing the `.filter()` *before* the `.groupBy()` is a data engineering best practice. It shrinks the dataset early so Spark doesn't waste cluster resources grouping rows we don't care about.

In [5]:
# Filter down to the West region first, then group by category to find the mean sale amount
df_q4 = df.filter(F.col("region") == "West") \
          .groupBy("product_category") \
          .agg(F.avg("sale_amount").alias("avg_sale_amount"))

# Display the final aggregated averages
df_q4.show(truncate=False)

+----------------+----------------+
|product_category|avg_sale_amount |
+----------------+----------------+
|Office Supplies |116.422376910912|
|Furniture       |357.302324611033|
|Technology      |420.687532554257|
+----------------+----------------+



### Q5: Handling Nulls — .na.drop() vs .na.fill()
Real datasets almost never come perfectly clean, and Spark gives us two very different strategies to manage missing values:

*   `.na.drop()` completely discards any row containing a null value in the specified column. It is useful when a row is completely unusable without that field, but you risk losing other valuable data in the process.
*   `.na.fill()` replaces the null value with a default fallback value you choose. This keeps the row intact so you don't lose the remaining good data.

Since a missing `status` doesn't make the entire sale record invalid, filling it with `'Unknown'` is the much safer approach.

In [6]:
# Check how many rows are missing a status before the operation
null_status_before = df.filter(F.col("status").isNull()).count()
print(f"Rows with null status before fill: {null_status_before}")

# Apply .na.fill using a target dictionary to avoid modifying other columns accidentally
df_filled = df.na.fill({"status": "Unknown"})

# Display a preview to verify the placeholder replacement
df_filled.select("user_id", "status").show(10)

Rows with null status before fill: 493
+--------+--------+
| user_id|  status|
+--------+--------+
|CG-12520|  Active|
|CG-12520|  Active|
|DV-13045|  Active|
|SO-20335|  Active|
|SO-20335|  Active|
|BH-11710| Unknown|
|BH-11710|Inactive|
|BH-11710|  Active|
|BH-11710|Inactive|
|BH-11710|  Active|
+--------+--------+
only showing top 10 rows


### Q6: Grouped Counts with a "Having" Style Filter
This step replicates the SQL `HAVING` clause pattern: filtering groups *after* an aggregation has been completed, rather than filtering raw rows beforehand. 

We bundle all rows by `city`, calculate the order volumes per group using `.count()`, and then apply a post-aggregation `.filter()` to drop the long tail of low-volume towns. This ensures we isolate only high-volume primary distribution centers.

In [7]:
# Group by city, calculate row frequencies, and apply a post-aggregate filter threshold (> 100)
city_counts = df.groupBy("city").count()
high_volume_cities = city_counts.filter(F.col("count") > 100) \
                                 .orderBy(F.col("count").desc())

# Display the high-volume operational hubs
high_volume_cities.show(20, truncate=False)

# Analytical summary check for documentation pipeline
total_cities = df.select("city").distinct().count()
passing_cities = high_volume_cities.count()

print(f"Total distinct cities: {total_cities}")
print(f"Cities with more than 100 orders (Tier-1 Distribution Hubs): {passing_cities}")

+-------------+-----+
|city         |count|
+-------------+-----+
|New York City|915  |
|Los Angeles  |747  |
|Philadelphia |537  |
|San Francisco|510  |
|Seattle      |428  |
|Houston      |377  |
|Chicago      |314  |
|Columbus     |222  |
|San Diego    |170  |
|Springfield  |163  |
|Dallas       |157  |
|Jacksonville |125  |
|Detroit      |115  |
+-------------+-----+

Total distinct cities: 531
Cities with more than 100 orders (Tier-1 Distribution Hubs): 13


## Q7: How Immutability Affects Data Cleaning

Spark DataFrames are immutable — once created, a DataFrame's contents can never be changed in
place. Every "cleaning" operation you perform, like dropping a column, renaming one, or filling
nulls, doesn't modify the existing DataFrame at all. Instead, it returns a brand new DataFrame
object that represents the transformation, while the original stays exactly as it was.

This has a few real consequences for how you write cleaning code:

1. **You must reassign the result.** Writing `df.drop("status")` on its own does nothing useful —
   it creates a new DataFrame in memory and then throws it away, because nothing captured it.
   You have to write `df = df.drop("status")` (or assign it to a new variable) to actually keep
   the change.

2. **Chaining becomes the natural style.** Because every operation returns a new DataFrame, it's
   idiomatic in Spark to chain multiple cleaning steps together in one expression, like:
   `df.dropDuplicates().na.fill({"price": 0.0}).drop("raw_timestamp")`
   Each step in the chain hands off a new DataFrame to the next step.

3. **Nothing is destroyed, so debugging is safer.** Since the original `df` is untouched, you can
   always go back to it if a cleaning step produces unexpected results — you're not fighting to
   "undo" a mutation, you just reference the earlier variable.

4. **It works cleanly with Spark's lazy evaluation.** Because operations build up as a lineage of
   transformations rather than actually changing data immediately, Spark can wait until an action
   (like `.show()` or `.count()`) is called, then optimize the entire chain of cleaning steps at
   once via the Catalyst optimizer, rather than executing each step wastefully one at a time.

In short: immutability means "cleaning" in Spark is really about building a new, corrected
version of your DataFrame step by step, not editing the original in place.

### Q8: Multi-Condition Filtering
In enterprise deployments, business requirements often demand compound data filtering. Here we isolate rows where users fall in an inclusive youth demographic age bracket ($18 \le \text{age} \le 30$) AND belong to the 'Premium' subscription tier.

A critical PySpark nuance is wrapping each logical condition inside its own set of parentheses. Because Python treats the bitwise AND operator (`&`) with a higher evaluation precedence than comparison operators (like `>=` or `==`), omitting parentheses will introduce compilation parsing bugs.

In [8]:
# Apply the multi-condition filter using explicit boundary groupings
young_premium_users = df.filter(
    (F.col("age") >= 18) &
    (F.col("age") <= 30) &
    (F.col("subscription") == "Premium")
)

# Force computation to measure matching population scale
print(f"Users aged 18-30 with Premium subscription: {young_premium_users.count()}")

# Display sample lines to verify categorical data distribution
young_premium_users.select("user_id", "age", "subscription", "product_category").show(10)

Users aged 18-30 with Premium subscription: 851
+--------+---+------------+----------------+
| user_id|age|subscription|product_category|
+--------+---+------------+----------------+
|DV-13045| 24|     Premium| Office Supplies|
|ZD-21925| 28|     Premium| Office Supplies|
|TB-21520| 28|     Premium| Office Supplies|
|TB-21520| 18|     Premium| Office Supplies|
|MA-17560| 30|     Premium| Office Supplies|
|ES-14080| 22|     Premium| Office Supplies|
|DP-13000| 27|     Premium| Office Supplies|
|JM-15265| 18|     Premium|      Technology|
|TB-21055| 18|     Premium| Office Supplies|
|KD-16270| 18|     Premium| Office Supplies|
+--------+---+------------+----------------+
only showing top 10 rows


## Q9: Why Handle Nulls Before Aggregating

Null values can silently distort mathematical aggregations if you don't deal with them first.

Functions like `sum()` and `avg()` in Spark actually skip nulls by default rather than erroring
out — which sounds convenient, but it's dangerous because it changes what the aggregation
actually means without telling you. For example, if you call `avg()` on a price column that has
nulls, Spark divides by the count of NON-null values, not the total row count. That can make an
average look higher or lower than it should be, and two people running the "same" aggregation on
data with different null patterns can get numbers that aren't actually comparable.

There's also a correctness risk specific to `sum()`: if you intended a null to mean "this
transaction had zero value" but Spark just skips it instead of treating it as zero, your total
revenue figure will be understated — but silently, with no error to warn you.

That's why it's best practice to explicitly decide how nulls should be treated (fill them with
0, drop the rows, or fill with a business-meaningful default) BEFORE aggregating, rather than
letting Spark's default null-skipping behavior make that decision for you implicitly.

### Q10: Schema Modification (Casting & Renaming)
Our raw `raw_timestamp` column is loaded as a text string. To perform time-based calculations, chronological sorting, or window actions, Spark must interpret it as an actual timestamp data type.

We use `.withColumn()` to generate a new column named `event_time` by casting the string values, and drop the original `raw_timestamp` column to keep our data layout clean.

In [9]:
# Convert the string column to a proper TimestampType and drop the original text column
df_schema_mod = df.withColumn("event_time", F.col("raw_timestamp").cast(TimestampType())) \
                  .drop("raw_timestamp")

# Verify the changes by printing the schema and a data sample
df_schema_mod.printSchema()
df_schema_mod.select("user_id", "event_time").show(10, truncate=False)

root
 |-- user_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+--------+-------------------+
|user_id |event_time         |
+--------+-------------------+
|CG-12520|2016-11-08 12:00:00|
|CG-12520|2016-11-08 12:00:00|
|DV-13045|2016-06-12 12:00:00|
|SO-20335|2015-10-11 12:00:00|
|SO-20335|2015-10-11 12:00:00|
|BH-11710|2014-06-09 12:00:00|
|BH-11710|2014-06-09 12:00:00|
|BH-11710|2014-06-09 12:00:00|
|BH-11710|2014-06-09 12:00:00|
|BH-11710|2014-06-09 12:00:00|
+--------+------------

## Q11: The Shuffle Process During Grouping

When you run a `groupBy()` followed by an aggregation, Spark needs to bring together all rows
that share the same group key (e.g. all rows for "New York City") so it can compute a single
result for that group. The problem is that Spark distributes data across multiple partitions
(and, in a real cluster, across multiple physical machines), so rows belonging to the same group
are very likely scattered across different partitions to begin with.

A "shuffle" is the process of physically moving that data across the network (or across
partitions on disk, even in local mode) so that all rows with the same key end up together on
the same partition, ready to be aggregated.

This is why grouping is called a "wide transformation": a wide transformation is any operation
where computing each output partition requires data from MULTIPLE input partitions. Contrast
this with a "narrow transformation" like `filter()` or `withColumn()`, where each output
partition only ever needs data from a single corresponding input partition — no data has to move
anywhere.

Shuffles are expensive because they involve disk I/O, serialization, and network transfer,
making them one of the biggest performance bottlenecks in Spark jobs. This is exactly why
techniques like broadcast joins exist — they're designed specifically to avoid triggering a
shuffle when one side of a join is small enough to just copy everywhere instead.

### Q12: Cleaning Missing Emails & Empty Usernames
This step addresses two different forms of invalid data: a true `null` value in the `email` column, and an empty string (`""`) in the `username` field. 

An empty string technically "exists" as data, so standard null-checking functions like `isNull()` will skip it. We use a combination of inequality conditions to clean out both invalid entries at the same time.

In [10]:
# Filter out null emails and ensure usernames are neither null nor empty strings
df_clean_users = df.filter(
    F.col("email").isNotNull() &
    (F.col("username") != "") &
    F.col("username").isNotNull()
)

# Measure the row count difference to verify the cleaning step
before = df.count()
after = df_clean_users.count()
print(f"Rows before cleaning: {before}")
print(f"Rows after cleaning:  {after}")
print(f"Total invalid rows removed: {before - after}")

Rows before cleaning: 9994
Rows after cleaning:  9648
Total invalid rows removed: 346


### Q13: Multi-Statistic Parallel Aggregations
Instead of running three separate queries to extract individual metrics, we use the `.agg()` builder to calculate the minimum, maximum, and average values of the `price` column simultaneously. This allows Spark to calculate all metrics in a single pass over the dataset instead of scanning the rows multiple times.

In [11]:
# Compute multiple summary statistics in a single query pass
df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.mean("price").alias("mean_price")
).show()

+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|    0.444| 22638.48|229.37077752449235|
+---------+---------+------------------+



## Q14: The Risk of inferSchema=true with Messy Date Formats

`inferSchema=True` is convenient — Spark scans a sample of the data and guesses each column's
type automatically, saving you from manually declaring a schema. But it works by pattern-
matching values, and dates are one of the easiest types to get wrong this way.

The core risk is that Spark's schema inference expects a REASONABLY consistent format across the
column. If your source data mixes formats — say some rows have `11/8/2016`, others have
`2016-11-08`, and a few have something malformed like `Nov 8 16` or a blank string — Spark can't
confidently infer a single date or timestamp type for the whole column. In that situation, it
typically falls back to inferring the column as a plain StringType instead of a proper date type.

That's dangerous specifically because it fails silently. Your notebook won't throw an error —
it'll just quietly give you a string column where you expected a date column. Then, days or
weeks later, when you try to do date arithmetic, sort chronologically, or filter by date range,
the operations either fail with a confusing type error, or worse, silently produce wrong results
because Spark is comparing dates as if they were plain text (where, for example, the string
"2/1/2020" alphabetically sorts before "10/1/2019", even though it's a later date).

A second, subtler risk: even when inference does pick TimestampType, if a few rows have a
genuinely broken date string, Spark will typically convert those specific values to null rather
than erroring out — meaning inconsistent formatting can quietly manufacture nulls in your dataset
that weren't there in the original source.

The safer practice in production pipelines is to explicitly define the schema yourself (using
`StructType`) rather than relying on inference, especially for date/timestamp columns — that way
type mismatches surface as clear errors you can fix, instead of silent, hard-to-trace bugs
downstream.

### Q15: The Final End-to-End Pipeline

This is where everything comes together. A real data pipeline isn't a single operation — it's a
sequence of transformations chained together, where the output of one step becomes the input to
the next. Because Spark DataFrames are immutable (see Q7), we can chain all of these
transformations in one continuous expression without ever losing track of intermediate state.

The three steps here, in order:
1. Remove duplicate rows across the full row (not just specific columns this time), so we don't
   double-count revenue from an accidental duplicate record.
2. Fill any null `price` values with 0, so a missing price doesn't silently vanish from the
   sum — a null would just get skipped by `sum()`, but a 0 correctly represents "no revenue
   contribution" while keeping the row.
3. Group everything by `store_id` and sum the (now null-free) price column to get total revenue
   per store — a genuinely useful business metric.

**Implementation note:** Since the final result here is small (one row per store), we convert it
to pandas using `.toPandas()` and save it with pandas' own CSV writer, rather than Spark's native
`.write().csv()`. This avoids a Windows-specific dependency (`winutils.exe`/`HADOOP_HOME`) that
Spark's file writer needs but isn't installed by default, and is a normal choice for small,
already-aggregated output in real pipelines.



In [14]:
# Step 1: dropDuplicates() with no arguments removes rows identical across every column
# Step 2: na.fill() replaces nulls in price with 0.0 so revenue math stays correct
# Step 3: groupBy + agg(sum) computes total revenue per store
final_pipeline_df = df \
    .dropDuplicates() \
    .na.fill({"price": 0.0}) \
    .groupBy("store_id") \
    .agg(F.sum("price").alias("total_revenue")) \
    .orderBy(F.col("total_revenue").desc())

final_pipeline_df.show(10, truncate=False)

+--------+------------------+
|store_id|total_revenue     |
+--------+------------------+
|ST_10035|76246.27499999995 |
|ST_10024|76014.93000000005 |
|ST_10009|54383.01199999998 |
|ST_94122|51848.857         |
|ST_10011|45346.26000000002 |
|ST_98105|41146.32999999999 |
|ST_98115|40908.97800000002 |
|ST_19134|38627.809         |
|ST_90049|37544.723999999966|
|ST_90045|36998.92299999999 |
+--------+------------------+
only showing top 10 rows


In [15]:
# Convert to pandas for the final small output — safely avoids Windows Hadoop/winutils issues
output_dir = os.path.join("..", "output")
os.makedirs(output_dir, exist_ok=True)

pandas_result = final_pipeline_df.toPandas()
pandas_result.to_csv(os.path.join(output_dir, "results.csv"), index=False)

print(f"Pipeline executed and results saved to {output_dir}/results.csv successfully!")

spark.stop()

Pipeline executed and results saved to ..\output/results.csv successfully!
